In [ ]:
def main(datasources, start_date, end_date):
    """生成格式联调因子，按参考模板返回 date、instrument、factor。"""
    import hashlib
    import re
    from datetime import date as date_type, datetime

    try:
        import dai
    except ModuleNotFoundError as error:
        # 只处理 dai 本身缺失，不掩盖平台 dai 内部依赖出错。
        if error.name != "dai":
            raise
        import sys
        from pathlib import Path

        current = Path.cwd().resolve()
        project = next(
            (
                path
                for path in (current, *current.parents)
                if (path / "dai.py").is_file()
                and (path / "docs" / "bigquant-2026-end-to-end").is_dir()
            ),
            None,
        )
        if project is None:
            raise ModuleNotFoundError(
                "请从 jycapital 项目或其子目录运行，并安装 requirements-local.txt。",
                name="dai",
            ) from error
        sys.path.insert(0, str(project))
        import dai
    import numpy as np
    import pandas as pd

    seed = 20261009

    def as_local_time(value):
        timestamp = pd.Timestamp(value)
        if pd.isna(timestamp):
            raise ValueError("评测时间不能为空。")
        if timestamp.tzinfo is not None:
            timestamp = timestamp.tz_convert("Asia/Shanghai").tz_localize(None)
        return timestamp

    start = as_local_time(start_date)
    end = as_local_time(end_date)
    end_is_date_only = (
        isinstance(end_date, str)
        and re.fullmatch(r"\d{4}-\d{2}-\d{2}", end_date.strip()) is not None
    ) or (isinstance(end_date, date_type) and not isinstance(end_date, datetime))
    if end_is_date_only:
        stop = end.normalize() + pd.Timedelta(days=1)
        if start >= stop:
            raise ValueError("start_date 必须早于 end_date 的次日零点。")
    else:
        if start > end:
            raise ValueError("start_date 不能晚于 end_date。")
        stop = end.floor("min") + pd.Timedelta(minutes=1)
    first = start.ceil("min")
    if first >= stop:
        raise ValueError("指定区间内没有完整的分钟采样时刻。")

    # Mock 只需要股票池的时间与股票键，不需要模型、标签或行情值。
    # 与参考模板一样保留 DAI 返回的键类型，不强制 date 为纳秒，
    # 不把 instrument 整列强制转换成 Python 字符串。
    stk = dai.query(
        f"""
            SELECT date, instrument FROM cpt_jyc_2026_instruments
            WHERE date >= '{first.strftime("%Y-%m-%d %H:%M:%S")}'
              AND date < '{stop.strftime("%Y-%m-%d %H:%M:%S")}'
        """,
        filters={
            "date": [
                first.normalize().strftime("%Y-%m-%d"),
                stop.ceil("D").strftime("%Y-%m-%d"),
            ]
        },
    ).df()
    if not {"date", "instrument"}.issubset(stk.columns):
        raise ValueError("股票池查询必须返回 date 和 instrument。")
    stk = stk[["date", "instrument"]].copy()
    if stk.empty:
        raise ValueError("评测区间内股票池为空。")

    # 用确定性 mock 替代参考模板的模型预测，仅 factor 转为 float64。
    idx_df = stk.drop_duplicates(["date", "instrument"]).copy()
    idx_df["date"] = pd.to_datetime(idx_df["date"])
    if idx_df[["date", "instrument"]].isna().any().any():
        raise ValueError("股票池包含空键。")
    timestamps = idx_df["date"].dt.strftime("%Y-%m-%dT%H:%M:%S")

    def mock_factors():
        for timestamp, instrument in zip(timestamps, idx_df["instrument"]):
            payload = f"{seed}|{timestamp}|{instrument}".encode("utf-8")
            digest = hashlib.blake2b(payload, digest_size=8).digest()
            integer = int.from_bytes(digest, "big") >> 11
            yield integer * (2.0 / (1 << 53)) - 1.0

    idx_df["factor"] = np.fromiter(
        mock_factors(), dtype=np.float64, count=len(idx_df)
    )

    # 逐项遵循用户提供的参考模板：完整时间内连接、清理、选列和排序。
    idx_df["date"] = pd.to_datetime(idx_df["date"])
    stk["date"] = pd.to_datetime(stk["date"])
    result = (
        pd.merge(stk, idx_df, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["factor"])
        .drop_duplicates(["date", "instrument"])[["date", "instrument", "factor"]]
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )
    print(
        f"FORMAT MOCK ONLY | rows={len(result)} | "
        f"dates={result['date'].nunique()} | "
        f"instruments={result['instrument'].nunique()}"
    )
    print("factor_data.dtypes:", result.dtypes.astype(str).to_dict())
    return result

if __name__ == "__main__":
    result = main({}, "2020-01-20", "2020-02-04")
    print(result.dtypes)
    print(result.shape)
    result.head()

